<a href="https://colab.research.google.com/github/jcbones01-glitch/ai-crypto-trading-command-center/blob/main/15m_VWAP_Backtest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import requests
import pandas as pd
import numpy as np

# 1. Fetch 1,000 15-minute BTCUSDT candles from Binance's unrestricted public data endpoint
print("Downloading 15m BTCUSDT data...")
url = "https://data-api.binance.vision/api/v3/klines"
params = {"symbol": "BTCUSDT", "interval": "15m", "limit": 1000}
raw_data = requests.get(url, params=params).json()

# 2. Build clean OHLCV DataFrame
cols = ['timestamp', 'open', 'high', 'low', 'close', 'volume', 'close_time', 'qav', 'trades', 'tbb', 'tbq', 'ignore']
df = pd.DataFrame(raw_data, columns=cols)
df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')
for col in ['open', 'high', 'low', 'close', 'volume']:
    df[col] = df[col].astype(float)

# 3. Calculate Daily Resetting VWAP & Standard Deviation Bands
df['date'] = df['timestamp'].dt.date
df['hlc3'] = (df['high'] + df['low'] + df['close']) / 3
df['pv'] = df['hlc3'] * df['volume']

# Group by UTC day so VWAP resets every session like TradingView
df['cum_vol'] = df.groupby('date')['volume'].cumsum()
df['cum_pv'] = df.groupby('date')['pv'].cumsum()
df['vwap'] = df['cum_pv'] / df['cum_vol']

df['sq_diff_vol'] = ((df['hlc3'] - df['vwap']) ** 2) * df['volume']
df['cum_sq_diff'] = df.groupby('date')['sq_diff_vol'].cumsum()
df['vwap_std'] = np.sqrt(df['cum_sq_diff'] / df['cum_vol'])

df['Upper_Band_1'] = df['vwap'] + df['vwap_std']
df['Lower_Band_1'] = df['vwap'] - df['vwap_std']
df['Upper_Band_2'] = df['vwap'] + (2 * df['vwap_std'])
df['Lower_Band_2'] = df['vwap'] - (2 * df['vwap_std'])

# Calculate 20-period volume moving average
df['vol_ma20'] = df['volume'].rolling(window=20).mean()

# 4. Detect 15m "Weakness" Rejection Wicks at Band 2 with Volume Filter
# Short trigger: High pierces Upper Band 2, candle closes back below it, and volume > 20 MA
df['Short_Trigger'] = (df['high'] > df['Upper_Band_2']) & (df['close'] < df['Upper_Band_2']) & (df['volume'] > df['vol_ma20'])
# Long trigger: Low pierces Lower Band 2, candle closes back above it, and volume > 20 MA
df['Long_Trigger'] = (df['low'] < df['Lower_Band_2']) & (df['close'] > df['Lower_Band_2']) & (df['volume'] > df['vol_ma20'])

# 5. Simulate 2:1 Reward-to-Risk Trades using the Rejection Wick Box
trades = []
for i in range(len(df) - 20):
    row = df.iloc[i]

    # Evaluate Short Setups
    if row['Short_Trigger']:
        entry = row['close']
        stop_loss = row['high']
        risk = stop_loss - entry
        if risk <= 0:
            continue
        target = entry - (2 * risk)  # 2R Target

        for j in range(i + 1, i + 20):
            future = df.iloc[j]
            if future['high'] >= stop_loss:
                trades.append({'Time': row['timestamp'], 'Type': 'SHORT', 'Result': -1.0})
                break
            elif future['low'] <= target:
                trades.append({'Time': row['timestamp'], 'Type': 'SHORT', 'Result': 2.0})
                break

    # Evaluate Long Setups
    elif row['Long_Trigger']:
        entry = row['close']
        stop_loss = row['low']
        risk = entry - stop_loss
        if risk <= 0:
            continue
        target = entry + (2 * risk)  # 2R Target

        for j in range(i + 1, i + 20):
            future = df.iloc[j]
            if future['low'] <= stop_loss:
                trades.append({'Time': row['timestamp'], 'Type': 'LONG', 'Result': -1.0})
                break
            elif future['high'] >= target:
                trades.append({'Time': row['timestamp'], 'Type': 'LONG', 'Result': 2.0})
                break

# 6. Print Backtest Summary
res_df = pd.DataFrame(trades)
if not res_df.empty:
    wins = len(res_df[res_df['Result'] > 0])
    total = len(res_df)
    win_rate = (wins / total) * 100
    net_r = res_df['Result'].sum()
    print(f"\n=== 15M VWAP STRATEGY RESULTS (Last 1,000 Candles) ===")
    print(f"Total Completed Trades: {total}")
    print(f"Win Rate: {win_rate:.1f}% (Break-even needed at 2R: 33.3%)")
    print(f"Net Return: {net_r:+.1f} R")
    print("\nLast 5 Trades:")
    print(res_df.tail(5).to_string(index=False))
else:
    print("No completed trades found in the sample window.")


=== 15M VWAP STRATEGY RESULTS (Last 1,000 Candles) ===
Total Completed Trades: 79
Win Rate: 35.4% (Break-even needed at 2R: 33.3%)
Net Return: +5.0 R

Last 5 Trades:
               Time Type  Result
2026-11-14 03:45:00 LONG    -1.0
2026-11-14 04:00:00 LONG     2.0
2026-11-14 14:00:00 LONG    -1.0
2026-11-14 15:30:00 LONG    -1.0
2026-11-14 16:45:00 LONG     2.0


In [3]:
import requests
import pandas as pd
import numpy as np
import xgboost as xgb

# 1. Fetch 3,000 15m BTCUSDT candles via pagination (Binance max=1000 per request)
print("Fetching 3,000 15m BTCUSDT candles...")
all_raw_data = []
end_time = None
url = "https://data-api.binance.vision/api/v3/klines"

for _ in range(3):
    params = {"symbol": "BTCUSDT", "interval": "15m", "limit": 1000}
    if end_time:
        params["endTime"] = end_time
    res = requests.get(url, params=params).json()
    all_raw_data = res + all_raw_data  # Prepend the older data
    end_time = res[0][0] - 1  # Set end_time to just before the oldest candle in this batch

# 2. Build OHLCV DataFrame
cols = ['timestamp', 'open', 'high', 'low', 'close', 'volume', 'close_time', 'qav', 'trades', 'tbb', 'tbq', 'ignore']
df = pd.DataFrame(all_raw_data, columns=cols)
df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')
for col in ['open', 'high', 'low', 'close', 'volume']:
    df[col] = df[col].astype(float)

# 3. Calculate VWAP & Bands
df['date'] = df['timestamp'].dt.date
df['hlc3'] = (df['high'] + df['low'] + df['close']) / 3
df['pv'] = df['hlc3'] * df['volume']

df['cum_vol'] = df.groupby('date')['volume'].cumsum()
df['cum_pv'] = df.groupby('date')['pv'].cumsum()
df['vwap'] = df['cum_pv'] / df['cum_vol']

df['sq_diff_vol'] = ((df['hlc3'] - df['vwap']) ** 2) * df['volume']
df['cum_sq_diff'] = df.groupby('date')['sq_diff_vol'].cumsum()
df['vwap_std'] = np.sqrt(df['cum_sq_diff'] / df['cum_vol'])

df['Upper_Band_2'] = df['vwap'] + (2 * df['vwap_std'])
df['Lower_Band_2'] = df['vwap'] - (2 * df['vwap_std'])

# 4. Feature Engineering for ML
df['vol_ma20'] = df['volume'].rolling(window=20).mean()
df['vol_ratio'] = df['volume'] / (df['vol_ma20'] + 1e-8)

# True Range & ATR (14)
df['prev_close'] = df['close'].shift(1)
df['tr'] = np.maximum(df['high'] - df['low'],
                      np.maximum(abs(df['high'] - df['prev_close']), abs(df['low'] - df['prev_close'])))
df['atr'] = df['tr'].rolling(14).mean()

# RSI (14)
delta = df['close'].diff()
gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)
avg_gain = gain.ewm(com=13, adjust=False).mean()
avg_loss = loss.ewm(com=13, adjust=False).mean()
rs = avg_gain / avg_loss
df['rsi'] = 100 - (100 / (1 + rs))

# Wick to Body Ratios
df['body'] = abs(df['close'] - df['open'])
df['upper_wick'] = df['high'] - df[['open', 'close']].max(axis=1)
df['lower_wick'] = df[['open', 'close']].min(axis=1) - df['low']
df['upper_wick_to_body'] = df['upper_wick'] / (df['body'] + 1e-8)
df['lower_wick_to_body'] = df['lower_wick'] / (df['body'] + 1e-8)

# Triggers
df['Short_Trigger'] = (df['high'] > df['Upper_Band_2']) & (df['close'] < df['Upper_Band_2']) & (df['volume'] > df['vol_ma20'])
df['Long_Trigger'] = (df['low'] < df['Lower_Band_2']) & (df['close'] > df['Lower_Band_2']) & (df['volume'] > df['vol_ma20'])

# 5. Simulate Strategy (2:1 Reward-to-Risk, 0.08R fee)
trades = []
fee = 0.08
for i in range(len(df) - 20):
    row = df.iloc[i]

    # Skip if features aren't warmed up
    if pd.isna(row['atr']) or pd.isna(row['rsi']) or pd.isna(row['vwap_std']):
        continue

    trade = None
    if row['Short_Trigger']:
        entry = row['close']
        risk = row['high'] - entry
        if risk > 0:
            target = entry - (2 * risk)
            for j in range(i + 1, i + 20):
                future = df.iloc[j]
                if future['high'] >= row['high']:
                    trade = {'Time': row['timestamp'], 'Type': 'SHORT', 'Result': -1.0 - fee}
                    break
                elif future['low'] <= target:
                    trade = {'Time': row['timestamp'], 'Type': 'SHORT', 'Result': 2.0 - fee}
                    break

    elif row['Long_Trigger']:
        entry = row['close']
        risk = entry - row['low']
        if risk > 0:
            target = entry + (2 * risk)
            for j in range(i + 1, i + 20):
                future = df.iloc[j]
                if future['low'] <= row['low']:
                    trade = {'Time': row['timestamp'], 'Type': 'LONG', 'Result': -1.0 - fee}
                    break
                elif future['high'] >= target:
                    trade = {'Time': row['timestamp'], 'Type': 'LONG', 'Result': 2.0 - fee}
                    break

    if trade:
        trade.update({
            'atr': row['atr'],
            'rsi': row['rsi'],
            'vol_ratio': row['vol_ratio'],
            'upper_wick_to_body': row['upper_wick_to_body'],
            'lower_wick_to_body': row['lower_wick_to_body']
        })
        trades.append(trade)

res_df = pd.DataFrame(trades)

# 6. ML XGBoost Filter
if len(res_df) < 10:
    print("Not enough trades found to train and test the model.")
else:
    print(f"\nTotal Trades found across 3000 candles: {len(res_df)}")

    # Chronological Split 70/30
    split_idx = int(len(res_df) * 0.7)
    train = res_df.iloc[:split_idx].copy()
    test = res_df.iloc[split_idx:].copy()

    features = ['atr', 'rsi', 'vol_ratio', 'upper_wick_to_body', 'lower_wick_to_body']

    X_train = train[features]
    y_train = (train['Result'] > 0).astype(int)  # 1 for Win, 0 for Loss
    X_test = test[features]
    y_test = (test['Result'] > 0).astype(int)

    # Train XGBoost Classifier
    model = xgb.XGBClassifier(n_estimators=100, max_depth=3, learning_rate=0.05, random_state=42)
    model.fit(X_train, y_train)

    # Predictions on Out-of-Sample Test set
    test['ML_Signal'] = model.predict(X_test)

    # Out-of-Sample Base Performance (Before Filter)
    base_wins = len(test[test['Result'] > 0])
    base_total = len(test)
    base_wr = (base_wins / base_total) * 100 if base_total > 0 else 0
    base_net_r = test['Result'].sum()

    # Out-of-Sample Filtered Performance (After XGBoost predicts Win)
    filt_test = test[test['ML_Signal'] == 1]
    filt_wins = len(filt_test[filt_test['Result'] > 0])
    filt_total = len(filt_test)
    filt_wr = (filt_wins / filt_total) * 100 if filt_total > 0 else 0
    filt_net_r = filt_test['Result'].sum()

    print("\n=== OUT-OF-SAMPLE TEST SET RESULTS (Last 30% of Trades) ===")
    print("--- BEFORE XGBoost Filter ---")
    print(f"Trades Taken: {base_total}")
    print(f"Win Rate: {base_wr:.1f}%")
    print(f"Net Return: {base_net_r:+.2f} R")

    print("\n--- AFTER XGBoost Filter ---")
    print(f"Trades Taken: {filt_total}")
    print(f"Win Rate: {filt_wr:.1f}%")
    print(f"Net Return: {filt_net_r:+.2f} R")

Fetching 3,000 15m BTCUSDT candles...

Total Trades found across 3000 candles: 252

=== OUT-OF-SAMPLE TEST SET RESULTS (Last 30% of Trades) ===
--- BEFORE XGBoost Filter ---
Trades Taken: 76
Win Rate: 36.8%
Net Return: +1.92 R

--- AFTER XGBoost Filter ---
Trades Taken: 16
Win Rate: 37.5%
Net Return: +0.72 R
